# Shakespeare task

## Import libraries and load the training data

In this section, the required libraries are imported and the Tiny Shakespeare dataset is loaded from the local `tinyshakespeare` folder. The length of the text is then printed to confirm that the dataset has been read successfully.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Read the text from the Tiny Shakespeare dataset
with open("tinyshakespeare/input.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Print length
print("Length of text:", len(text))

# Preview first 300 characters
print("\nPreview of text:\n")
print(text[:300])

Length of text: 1115394

Preview of text:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Character encoding

In this section, the unique characters in the Tiny Shakespeare dataset are identified. Each character is mapped to an integer index, and dictionaries are created for both character-to-index and index-to-character conversion.

In [2]:
# Get all unique characters in the text
chars = sorted(list(set(text)))
num_chars = len(chars)

# Create character-to-index and index-to-character mappings
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print("Number of unique characters:", num_chars)
print("First 20 characters:", chars[:20])

Number of unique characters: 65
First 20 characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G']


## Creating input sequences

In this section, fixed-length character sequences are created from the text. Each input sequence is paired with the next character as the target output.

In [3]:
# Define sequence length
seq_length = 40

# Create input and target sequences
sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, 3):
    sentences.append(text[i:i + seq_length])
    next_chars.append(text[i + seq_length])

print("Number of sequences:", len(sentences))

Number of sequences: 371785


## Vectorizing the sequences

In this section, the input sequences are vectorized into binary matrices and the target characters are one-hot encoded. This prepares the data in the format required for model training.

In [4]:
import numpy as np

# Vectorize inputs and targets
X = np.zeros((len(sentences), seq_length, num_chars), dtype=np.float32)
y = np.zeros((len(sentences), num_chars), dtype=np.float32)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_ix[char]] = 1
    y[i, char_to_ix[next_chars[i]]] = 1

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (371785, 40, 65)
Target shape: (371785, 65)


## Converting data to PyTorch tensors

In this section, the vectorized input and target data are converted into PyTorch tensors for training the recurrent neural network.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(np.argmax(y, axis=1), dtype=torch.long)

print("X tensor shape:", X_tensor.shape)
print("y tensor shape:", y_tensor.shape)

X tensor shape: torch.Size([371785, 40, 65])
y tensor shape: torch.Size([371785])


## Building the RNN model

In this section, a simple recurrent neural network is defined using PyTorch. The model takes character sequences as input and predicts the next character in the sequence.

In [6]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

hidden_size = 128
model = CharRNN(num_chars, hidden_size, num_chars)

print(model)

CharRNN(
  (rnn): RNN(65, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=65, bias=True)
)


## Defining loss function and optimizer

In this section, the loss function and optimizer are defined. The loss function measures prediction error, and the optimizer updates the model weights during training.

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training the RNN model

In this section, the recurrent neural network is trained for 10 epochs using a subset of the dataset to reduce training time.

In [8]:
# Use a subset for faster training
subset_size = 5000

X_train = X_tensor[:subset_size]
y_train = y_tensor[:subset_size]

epochs = 10
batch_size = 64

for epoch in range(epochs):
    total_loss = 0.0

    for i in range(0, len(X_train), batch_size):
        inputs = X_train[i:i + batch_size]
        targets = y_train[i:i + batch_size]

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 270.0907
Epoch 2, Loss: 255.9630
Epoch 3, Loss: 245.7776
Epoch 4, Loss: 228.3939
Epoch 5, Loss: 213.2555
Epoch 6, Loss: 201.7623
Epoch 7, Loss: 193.1355
Epoch 8, Loss: 186.4501
Epoch 9, Loss: 180.8883
Epoch 10, Loss: 176.1544


## Generating sample text

In this section, a function is created to generate 100 words of text using the trained model. A seed sequence is used to start the generation process.

In [10]:
def generate_text(model, seed_text, word_count=100):
    model.eval()
    generated = seed_text

    with torch.no_grad():
        while len(generated.split()) < word_count:
            input_seq = generated[-seq_length:]

            if len(input_seq) < seq_length:
                input_seq = " " * (seq_length - len(input_seq)) + input_seq

            x_pred = np.zeros((1, seq_length, num_chars), dtype=np.float32)

            for t, char in enumerate(input_seq):
                if char in char_to_ix:
                    x_pred[0, t, char_to_ix[char]] = 1

            x_pred_tensor = torch.tensor(x_pred, dtype=torch.float32)
            output = model(x_pred_tensor)

            predicted_index = torch.argmax(output, dim=1).item()
            next_char = ix_to_char[predicted_index]

            generated += next_char

    return " ".join(generated.split()[:word_count])


seed_text = text[:40]
generated_text = generate_text(model, seed_text, word_count=100)

print("Generated text:\n")
print(generated_text)

Generated text:

First Citizen: Before we proceed any fure the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere the ere t


## Conclusion

In this task, a recurrent neural network was implemented using the Tiny Shakespeare dataset. The text was preprocessed into character sequences, vectorized, and used to train the model for next-character prediction. A sample of generated text was then produced to demonstrate the model’s learned language patterns.